# 🚀 ViniVideo AI — Motor grátis no Colab

Este notebook inicia um backend de vídeo para o app Android usando uma GPU do Google Colab.

**Antes de rodar:** em **Ambiente de execução → Alterar tipo de ambiente de execução**, escolha **GPU T4**. Depois use **Executar tudo**.

O primeiro início pode demorar porque o modelo precisa ser baixado. O Colab gratuito pode limitar ou encerrar a sessão; por isso este motor é para testes e não é realmente ilimitado.


In [ ]:
import torch
print("CUDA disponível:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Ative uma GPU T4 no Colab antes de continuar.")
print("GPU:", torch.cuda.get_device_name(0))


In [ ]:
!pip -q install --upgrade-strategy only-if-needed "diffusers==0.33.1" "transformers==4.51.3" "huggingface-hub==0.30.2" "tokenizers>=0.21,<0.22" "accelerate>=1.2,<2" "sentencepiece>=0.2,<0.3" "safetensors>=0.4" fastapi uvicorn pillow imageio imageio-ffmpeg
!pip -q install "requests==2.32.4"
print("Dependências compatíveis instaladas ✓")


In [ ]:
import diffusers
import transformers
import huggingface_hub

print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)

from diffusers import LTXPipeline
print("LTXPipeline (texto→vídeo) importado ✓")

try:
    from diffusers import LTXImageToVideoPipeline
    print("LTXImageToVideoPipeline importado ✓")
except Exception as exc:
    print("⚠️ Image-to-video não carregou neste ambiente.")
    print(type(exc).__name__ + ":", str(exc)[:1200])
    print("O teste de 5 s sem imagem ainda pode funcionar com texto→vídeo.")


In [ ]:
%%writefile /content/vinivideo_server.py
import base64
import gc
import os
import shutil
import subprocess
import threading
import traceback
import uuid
from io import BytesIO
from pathlib import Path

import torch
from fastapi import BackgroundTasks, FastAPI, HTTPException
from fastapi.staticfiles import StaticFiles
from PIL import Image

OUTPUT_DIR = Path(os.environ.get("VINIVIDEO_OUTPUT_DIR", "/content/vinivideo_outputs"))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_ID = os.environ.get("VINIVIDEO_MODEL_ID", "Lightricks/LTX-Video")
NEGATIVE = (
    "worst quality, low quality, blurry, jittery, distorted anatomy, "
    "deformed face, duplicated subject, random cuts, flicker, text, watermark"
)

app = FastAPI(title="ViniVideo AI Free GPU Backend", version="0.3")
app.mount("/outputs", StaticFiles(directory=str(OUTPUT_DIR)), name="outputs")

JOBS = {}
JOBS_LOCK = threading.Lock()
MODEL_LOCK = threading.RLock()
TEXT_PIPE = None
IMAGE_PIPE = None
LAST_ERROR = ""
LOG_PATH = OUTPUT_DIR / "vinivideo-backend.log"


def gpu_name():
    if not torch.cuda.is_available():
        return "CPU"
    return torch.cuda.get_device_name(0)


def log_line(message):
    line = str(message)
    print(line, flush=True)
    try:
        with open(LOG_PATH, "a", encoding="utf-8") as f:
            f.write(line + "\n")
    except Exception:
        pass


def set_job(job_key, **changes):
    with JOBS_LOCK:
        current = JOBS.get(job_key, {})
        current.update(changes)
        JOBS[job_key] = current


def get_job(job_id):
    with JOBS_LOCK:
        return dict(JOBS.get(job_id, {}))


def load_text_pipeline():
    global TEXT_PIPE
    with MODEL_LOCK:
        if TEXT_PIPE is not None:
            return TEXT_PIPE

        if not torch.cuda.is_available():
            raise RuntimeError("GPU CUDA não disponível. No Colab, ative Runtime > Change runtime type > T4 GPU.")

        from diffusers import LTXPipeline

        set_dtype = torch.float16
        TEXT_PIPE = LTXPipeline.from_pretrained(
            MODEL_ID,
            torch_dtype=set_dtype,
        )
        try:
            TEXT_PIPE.vae.enable_tiling()
        except Exception:
            pass
        TEXT_PIPE.enable_model_cpu_offload()
        return TEXT_PIPE


def load_image_pipeline():
    global IMAGE_PIPE
    with MODEL_LOCK:
        if IMAGE_PIPE is not None:
            return IMAGE_PIPE

        text_pipe = load_text_pipeline()
        from diffusers import LTXImageToVideoPipeline

        IMAGE_PIPE = LTXImageToVideoPipeline.from_pipe(text_pipe)
        try:
            IMAGE_PIPE.vae.enable_tiling()
        except Exception:
            pass
        IMAGE_PIPE.enable_model_cpu_offload()
        return IMAGE_PIPE


def decode_reference(data_uri):
    if not data_uri:
        return None
    try:
        encoded = data_uri.split(",", 1)[1] if "," in data_uri else data_uri
        raw = base64.b64decode(encoded)
        return Image.open(BytesIO(raw)).convert("RGB")
    except Exception:
        return None


def aspect_size(aspect, quality):
    quality = str(quality or "qualidade").lower()
    if "cinema" in quality:
        long_side, short_side = 640, 352
    elif "qualidade" in quality:
        long_side, short_side = 576, 320
    else:
        long_side, short_side = 512, 288

    if aspect == "9:16":
        return short_side, long_side
    if aspect == "1:1":
        side = 448 if "cinema" in quality else 384
        return side, side
    return long_side, short_side


def render_profile(quality, scene_duration):
    q = str(quality or "qualidade").lower()
    if "cinema" in q:
        frames, steps = 81, 28
    elif "qualidade" in q:
        frames, steps = 65, 20
    else:
        frames, steps = 49, 12

    duration = max(1, int(scene_duration or 4))
    fps = max(8, min(24, round((frames - 1) / duration)))
    return frames, steps, fps


def save_scene(frames, output_path, fps):
    from diffusers.utils import export_to_video
    export_to_video(frames, str(output_path), fps=fps)


def stitch_videos(scene_paths, final_path):
    concat_file = final_path.with_suffix(".txt")
    with open(concat_file, "w", encoding="utf-8") as f:
        for path in scene_paths:
            safe = str(path).replace("'", "'\\''")
            f.write(f"file '{safe}'\n")

    cmd = [
        "ffmpeg", "-y",
        "-f", "concat", "-safe", "0",
        "-i", str(concat_file),
        "-c:v", "libx264",
        "-preset", "veryfast",
        "-crf", "18",
        "-pix_fmt", "yuv420p",
        "-movflags", "+faststart",
        str(final_path),
    ]
    subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    try:
        concat_file.unlink()
    except Exception:
        pass


def run_generation(job_id, payload):
    job_dir = OUTPUT_DIR / job_id
    job_dir.mkdir(parents=True, exist_ok=True)

    try:
        set_job(
            job_id,
            status="RUNNING",
            progress=2,
            stage="Preparando motor de vídeo",
        )

        text_pipe = load_text_pipeline()
        image_pipe = None

        aspect = payload.get("aspect_ratio", "9:16")
        quality = payload.get("quality_preset", "qualidade")
        width, height = aspect_size(aspect, quality)
        seed = int(payload.get("seed", 1))
        continuity = payload.get("continuity_mode", "") == "last_frame_and_reference"
        reference = decode_reference(payload.get("reference_image", ""))
        scenes = payload.get("scenes") or [{
            "index": 1,
            "duration_seconds": min(5, int(payload.get("duration_seconds", 5))),
            "prompt": payload.get("prompt", ""),
        }]

        scene_paths = []
        last_frame = reference

        for position, scene in enumerate(scenes):
            index = int(scene.get("index", position + 1))
            duration = max(1, min(6, int(scene.get("duration_seconds", 4))))
            prompt = str(scene.get("prompt") or payload.get("prompt") or "").strip()
            frames_count, steps, fps = render_profile(quality, duration)

            percent_start = 5 + int((position / max(1, len(scenes))) * 85)
            set_job(
                job_id,
                progress=percent_start,
                stage=f"Gerando tomada {position + 1}/{len(scenes)}",
            )

            generator = torch.Generator(device="cpu").manual_seed(seed + index)

            scene_base = 5 + int(
                (position / max(1, len(scenes))) * 85
            )
            scene_span = 85 / max(1, len(scenes))

            def on_step_end(
                pipeline,
                step,
                timestep,
                callback_kwargs,
            ):
                fraction = (step + 1) / max(1, steps)
                live_progress = min(
                    89,
                    int(scene_base + (scene_span * fraction * 0.92)),
                )
                set_job(
                    job_id,
                    progress=live_progress,
                    stage=(
                        f"Tomada {position + 1}/{len(scenes)} • "
                        f"passo {step + 1}/{steps}"
                    ),
                )
                return callback_kwargs

            common = dict(
                prompt=prompt,
                negative_prompt=NEGATIVE,
                width=width,
                height=height,
                num_frames=frames_count,
                num_inference_steps=steps,
                generator=generator,
                callback_on_step_end=on_step_end,
            )

            if last_frame is not None:
                if image_pipe is None:
                    set_job(
                        job_id,
                        stage="Carregando módulo de continuidade imagem→vídeo",
                    )
                    image_pipe = load_image_pipeline()
                image = last_frame.resize(
                    (width, height),
                    Image.Resampling.LANCZOS,
                )
                result = image_pipe(image=image, **common)
            else:
                result = text_pipe(**common)

            frames = result.frames[0]
            if not frames:
                raise RuntimeError(f"A tomada {index} não retornou frames.")

            scene_path = job_dir / f"scene_{index:03d}.mp4"
            save_scene(frames, scene_path, fps)
            scene_paths.append(scene_path)

            if continuity:
                last_frame = frames[-1].convert("RGB")
            elif reference is not None:
                last_frame = reference
            else:
                last_frame = None

            del result, frames
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

            percent_done = 5 + int(((position + 1) / len(scenes)) * 85)
            set_job(
                job_id,
                progress=percent_done,
                stage=f"Tomada {position + 1}/{len(scenes)} concluída",
            )

        set_job(job_id, progress=92, stage="Juntando as tomadas")
        final_path = OUTPUT_DIR / f"{job_id}.mp4"
        stitch_videos(scene_paths, final_path)

        if payload.get("generate_audio", False):
            audio_note = " • áudio IA não disponível no motor grátis"
        else:
            audio_note = ""

        set_job(
            job_id,
            status="DONE",
            progress=100,
            stage="Concluído" + audio_note,
            output_url=f"/outputs/{job_id}.mp4",
        )

        try:
            shutil.rmtree(job_dir)
        except Exception:
            pass

    except torch.cuda.OutOfMemoryError:
        global LAST_ERROR
        LAST_ERROR = "CUDA out of memory"
        log_line(f"[{job_id}] CUDA out of memory")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        set_job(
            job_id,
            status="ERROR",
            progress=0,
            stage="GPU sem memória. Tente o preset Rápido ou um vídeo menor.",
            error="CUDA out of memory",
        )
    except Exception as exc:
        LAST_ERROR = traceback.format_exc()[-4000:]
        log_line(f"[{job_id}] {LAST_ERROR}")
        set_job(
            job_id,
            status="ERROR",
            progress=0,
            stage=f"Falha: {str(exc)[:180]}",
            error=traceback.format_exc()[-4000:],
        )


@app.get("/api/v1/health")
def health():
    return {
        "status": "ok",
        "backend": "vinivideo-colab-free",
        "gpu": gpu_name(),
        "cuda": torch.cuda.is_available(),
        "model_profile": "ltx-video-2b",
        "model_id": MODEL_ID,
        "model_ready": TEXT_PIPE is not None,
        "capabilities": {
            "text_to_video": True,
            "image_to_video": True,
            "last_frame_continuity": True,
            "audio_generation": False,
        },
    }


@app.post("/api/v1/warmup")
def warmup():
    global LAST_ERROR
    try:
        load_text_pipeline()
        return {
            "status": "ok",
            "model_ready": True,
            "gpu": gpu_name(),
            "model_id": MODEL_ID,
        }
    except Exception as exc:
        LAST_ERROR = traceback.format_exc()[-4000:]
        log_line(LAST_ERROR)
        raise HTTPException(
            status_code=500,
            detail=f"{type(exc).__name__}: {str(exc)}",
        )


@app.post("/api/v1/jobs")
def create_job(payload: dict, background_tasks: BackgroundTasks):
    global LAST_ERROR
    try:
        if not torch.cuda.is_available():
            raise HTTPException(
                status_code=503,
                detail="CUDA não disponível. Ative uma GPU T4 no Colab.",
            )

        if payload.get("dry_run"):
            return {
                "job_id": "dry-run",
                "status": "DONE",
                "progress": 100,
                "stage": "API pronta",
                "output_url": "",
            }

        prompt = str(payload.get("prompt", "")).strip()
        if not prompt:
            raise HTTPException(
                status_code=400,
                detail="O prompt está vazio.",
            )

        job_id = uuid.uuid4().hex
        set_job(
            job_id,
            job_id=job_id,
            status="QUEUED",
            progress=1,
            stage="Na fila",
            output_url="",
        )

        background_tasks.add_task(run_generation, job_id, payload)
        log_line(f"[{job_id}] job aceito")
        return get_job(job_id)

    except HTTPException:
        raise
    except Exception as exc:
        LAST_ERROR = traceback.format_exc()[-4000:]
        log_line(LAST_ERROR)
        raise HTTPException(
            status_code=500,
            detail=f"{type(exc).__name__}: {str(exc)}",
        )


@app.get("/api/v1/jobs/{job_id}")
def job_status(job_id: str):
    job = get_job(job_id)
    if not job:
        raise HTTPException(status_code=404, detail="Job não encontrado")
    return job


@app.get("/api/v1/diagnostics")
def diagnostics():
    tail = ""
    try:
        if LOG_PATH.exists():
            lines = LOG_PATH.read_text(
                encoding="utf-8",
                errors="ignore",
            ).splitlines()
            tail = "\n".join(lines[-80:])
    except Exception:
        pass

    return {
        "status": "ok",
        "gpu": gpu_name(),
        "cuda": torch.cuda.is_available(),
        "jobs": len(JOBS),
        "last_error": LAST_ERROR,
        "log_tail": tail,
    }


@app.get("/")
def root():
    return {
        "name": "ViniVideo AI Free GPU Backend",
        "status": "online",
        "health": "/api/v1/health",
    }


In [ ]:
import os
import pathlib
import re
import socket
import subprocess
import sys
import time
import requests

# Encerra processos antigos desta sessão, se houver.
for proc_name in ["uvicorn", "cloudflared"]:
    subprocess.run(
        ["pkill", "-f", proc_name],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )

# Inicia a API local.
SERVER_LOG = open("/content/vinivideo-server.log", "w")
SERVER = subprocess.Popen(
    [
        sys.executable,
        "-m",
        "uvicorn",
        "vinivideo_server:app",
        "--host",
        "0.0.0.0",
        "--port",
        "8000",
    ],
    cwd="/content",
    stdout=SERVER_LOG,
    stderr=subprocess.STDOUT,
)

for _ in range(60):
    try:
        response = requests.get(
            "http://127.0.0.1:8000/api/v1/health",
            timeout=2,
        )
        if response.ok:
            print("Backend local online ✓")
            print(response.json())
            break
    except Exception:
        pass
    time.sleep(1)
else:
    raise RuntimeError(
        "O backend local não iniciou. Veja /content/vinivideo-server.log"
    )

# Baixa/carrega o modelo ANTES de abrir o túnel público.
# Isso evita que o app fique parado em 2% enquanto o primeiro job baixa o LTX.
print()
print("⏳ Preparando o LTX-Video na Tesla T4...")
print("Na primeira vez isso pode demorar alguns minutos.")
try:
    warmup = requests.post(
        "http://127.0.0.1:8000/api/v1/warmup",
        timeout=1800,
    )
    if not warmup.ok:
        raise RuntimeError(
            "Warmup HTTP "
            + str(warmup.status_code)
            + ": "
            + warmup.text[:2000]
        )
    print("✅ MODELO PRONTO NA GPU")
    print(warmup.json())
except Exception as exc:
    diagnostics = ""
    try:
        diagnostics = requests.get(
            "http://127.0.0.1:8000/api/v1/diagnostics",
            timeout=10,
        ).text
    except Exception:
        pass
    raise RuntimeError(
        "O modelo não conseguiu preparar antes de abrir o túnel. "
        + str(exc)
        + "\nDiagnóstico: "
        + diagnostics[:3000]
    )

# Baixa o Cloudflare Tunnel, sem cadastro.
CLOUDFLARED = "/content/cloudflared"
if not os.path.exists(CLOUDFLARED):
    subprocess.run(
        [
            "wget",
            "-q",
            "-O",
            CLOUDFLARED,
            "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
        ],
        check=True,
    )
    os.chmod(CLOUDFLARED, 0o755)

pattern = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
session = requests.Session()
session.trust_env = False

def start_public_tunnel(attempt):
    subprocess.run(
        ["pkill", "-f", "cloudflared"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    time.sleep(1)

    log_path = f"/content/cloudflared-{attempt}.log"
    tunnel_log = open(log_path, "w")
    tunnel = subprocess.Popen(
        [
            CLOUDFLARED,
            "tunnel",
            "--url",
            "http://127.0.0.1:8000",
            "--protocol",
            "http2",
            "--no-autoupdate",
        ],
        stdout=tunnel_log,
        stderr=subprocess.STDOUT,
    )

    public_url = None
    for _ in range(90):
        time.sleep(1)
        if tunnel.poll() is not None:
            break
        try:
            text = pathlib.Path(log_path).read_text(errors="ignore")
        except Exception:
            text = ""
        match = pattern.search(text)
        if match:
            public_url = match.group(0)
            break

    if not public_url:
        if tunnel.poll() is None:
            tunnel.terminate()
        return None, None, "Cloudflare não forneceu uma URL"

    host = public_url.removeprefix("https://")
    print(f"URL criada: {public_url}")
    print("Aguardando o DNS público ficar disponível...")

    last_error = ""
    for wait_step in range(60):
        if tunnel.poll() is not None:
            return None, None, "O processo cloudflared encerrou durante a conexão"

        try:
            socket.getaddrinfo(host, 443)
            health = session.get(
                public_url + "/api/v1/health",
                timeout=10,
            )
            if health.ok:
                return public_url, tunnel, None
            last_error = (
                f"Health HTTP {health.status_code}: "
                + health.text[:300]
            )
        except Exception as exc:
            last_error = f"{type(exc).__name__}: {exc}"

        if wait_step in (0, 9, 19, 29, 44):
            print(
                f"Ainda aguardando DNS/túnel... "
                f"({wait_step + 1}/60)"
            )
        time.sleep(2)

    if tunnel.poll() is None:
        tunnel.terminate()
    return None, None, last_error


def validate_public_api(public_url):
    health = session.get(
        public_url + "/api/v1/health",
        timeout=15,
    )
    if not health.ok:
        return False, f"health HTTP {health.status_code}: {health.text[:300]}"

    dry = session.post(
        public_url + "/api/v1/jobs",
        json={"dry_run": True, "prompt": "self-test"},
        timeout=20,
    )
    if not dry.ok:
        return False, f"jobs HTTP {dry.status_code}: {dry.text[:500]}"

    return True, None


public_url = None
TUNNEL = None
last_tunnel_error = ""

for attempt in range(1, 4):
    print(f"Criando túnel público — tentativa {attempt}/3...")
    public_url, TUNNEL, last_tunnel_error = start_public_tunnel(attempt)
    if public_url:
        ok, api_error = validate_public_api(public_url)
        if ok:
            break
        last_tunnel_error = api_error
        if TUNNEL is not None and TUNNEL.poll() is None:
            TUNNEL.terminate()
        public_url = None
        TUNNEL = None

    print("Esse túnel não ficou acessível; criando outro...")
    print(last_tunnel_error)

if not public_url or TUNNEL is None:
    raise RuntimeError(
        "Não consegui criar um túnel Cloudflare acessível após 3 tentativas. "
        + last_tunnel_error
    )

print("GET health ✓")
print("POST jobs (dry-run) ✓")
print()
print("=" * 70)
print("✅ MOTOR ONLINE")
print("Cole esta URL no app ViniVideo AI > Backend > URL do motor:")
print()
print(public_url)
print()
print("A célula ficará RODANDO enquanto o motor estiver ativo.")
print("Não pare esta célula durante a geração.")
print("=" * 70)

# Mantém esta célula ativa e vigia o túnel.
# Se o Quick Tunnel cair, um novo é criado automaticamente e a NOVA URL
# é impressa. Nesse caso, copie a nova URL para o app.
consecutive_failures = 0
watchdog_generation = 3

while True:
    time.sleep(15)

    if SERVER.poll() is not None:
        raise RuntimeError(
            "O backend local parou. Execute tudo novamente."
        )

    tunnel_dead = TUNNEL.poll() is not None
    public_ok = False
    public_error = ""

    if not tunnel_dead:
        try:
            health = session.get(
                public_url + "/api/v1/health",
                timeout=10,
            )
            public_ok = health.ok
            if not health.ok:
                public_error = (
                    f"HTTP {health.status_code}: "
                    + health.text[:250]
                )
        except Exception as exc:
            public_error = f"{type(exc).__name__}: {exc}"

    if public_ok:
        consecutive_failures = 0
        continue

    consecutive_failures += 1
    print(
        f"⚠️ Túnel instável ({consecutive_failures}/2): "
        + (public_error or "processo cloudflared encerrou")
    )

    if consecutive_failures < 2:
        continue

    print("♻️ Recriando o túnel automaticamente...")
    new_url = None
    new_tunnel = None
    last_error = ""

    for _ in range(3):
        watchdog_generation += 1
        new_url, new_tunnel, last_error = start_public_tunnel(
            watchdog_generation
        )
        if new_url:
            ok, api_error = validate_public_api(new_url)
            if ok:
                break
            last_error = api_error
            if new_tunnel is not None and new_tunnel.poll() is None:
                new_tunnel.terminate()
            new_url = None
            new_tunnel = None

    if not new_url or new_tunnel is None:
        raise RuntimeError(
            "O túnel caiu e não consegui criar outro. "
            + last_error
        )

    public_url = new_url
    TUNNEL = new_tunnel
    consecutive_failures = 0

    print()
    print("=" * 70)
    print("🔄 NOVA URL DO MOTOR — COPIE ESTA NO APP")
    print(public_url)
    print("=" * 70)


## Depois que aparecer a URL

1. Copie a URL `https://....trycloudflare.com`.
2. Volte ao app **ViniVideo AI**.
3. Abra **Backend**.
4. Cole em **URL do motor**.
5. Deixe **API key** vazia.
6. Toque em **Salvar** e depois **Testar conexão**.
7. Quando aparecer **Motor conectado**, abra **Criar** e gere um vídeo curto primeiro.

### Primeiro teste recomendado

Use **5 ou 10 segundos**, preset **Rápido**, e uma imagem de referência. Depois aumente a duração. Vídeos longos são montados em várias tomadas e podem levar bastante tempo numa GPU gratuita.
